# Week 2 - Data Processing with Python

# Step 1: Upload the 3 CSV files in Colab

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving customers.csv to customers.csv
Saving delivery_status.csv to delivery_status.csv
Saving orders.csv to orders.csv


# Step 2: Import Libraries and Load the Data

In [ ]:
import pandas as pd
import numpy as np

# load all CSVs
orders = pd.read_csv("orders.csv")
customers = pd.read_csv("customers.csv")
delivery_status = pd.read_csv("delivery_status.csv")

# preview
print("Orders:")
print(orders.head())
print("\nCustomers:")
print(customers.head())
print("\nDelivery Status:")
print(delivery_status.head())


Orders:
   order_id  customer_id  order_date delivery_date     status
0         1            1  2025-07-01    2025-07-03  Delivered
1         2            2  2025-07-05    2025-07-08  Delivered
2         3            3  2025-07-10    2025-07-12    Pending
3         4            4  2025-07-12    2025-07-15    Shipped
4         5            1  2025-07-18    2025-07-20    Pending

Customers:
   customer_id          name               email       phone   region
0            1   Rahul Kumar   rahul@example.com  9876543210    North
1            2  Anita Sharma   anita@example.com  9123456780    South
2            3     Vijay Rao   vijay@example.com  9988776655     East
3            4  Swathi Menon  swathi@example.com  9090909090     West
4            5     Arjun Das   arjun@example.com  9012345678  Central

Delivery Status:
   delivery_id  order_id current_status         last_updated
0            1         1      Delivered  2025-07-03 10:00:00
1            2         2      Delivered  2025-07

# Step 3: Check for Missing Values

In [ ]:
# check for missing values
print("\nMissing values in orders:\n", orders.isnull().sum())
print("\nMissing values in customers:\n", customers.isnull().sum())
print("\nMissing values in delivery_status:\n", delivery_status.isnull().sum())



Missing values in orders:
 order_id         0
customer_id      0
order_date       0
delivery_date    0
status           0
dtype: int64

Missing values in customers:
 customer_id    0
name           0
email          0
phone          0
region         0
dtype: int64

Missing values in delivery_status:
 delivery_id       0
order_id          0
current_status    0
last_updated      0
dtype: int64


# Step 4: Clean Missing Values


In [ ]:
# Drop rows with any missing values
orders.dropna(inplace=True)
customers.dropna(inplace=True)
delivery_status.dropna(inplace=True)




# Step 5: Convert Timestamps to Datetime Format

In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['delivery_date'] = pd.to_datetime(orders['delivery_date'])
delivery_status['last_updated'] = pd.to_datetime(delivery_status['last_updated'])


# Step 6: Calculate Delay in Days

In [ ]:
# Merge orders with delivery status
merged_df = pd.merge(orders, delivery_status, left_on='order_id', right_on='order_id', how='inner')

# Calculate delay in days: actual delivery - expected delivery
merged_df['delay_days'] = (merged_df['last_updated'] - merged_df['delivery_date']).dt.days

# If delay_days is negative, set it to 0
merged_df['delay_days'] = merged_df['delay_days'].apply(lambda x: x if x > 0 else 0)



# Step 7: Add a Flag Column (delayed)

In [ ]:
# Add delayed flag
merged_df["delayed"] = np.where(merged_df["delay_days"] > 0, 1, 0)



# Step 8: Top Delayed Customers


In [ ]:
# Merge with customers to get names
final_df = pd.merge(
    merged_df, customers,
    left_on="customer_id", right_on="customer_id",
    how="inner"
)

# Group by customer and count delays
delay_summary = final_df.groupby(["customer_id", "name"])["delayed"].sum().reset_index()

# Sort by number of delays (descending)
delay_summary = delay_summary.sort_values(by="delayed", ascending=False)

print("Top Delayed Customers:")
print(delay_summary)


Top Delayed Customers:
   customer_id          name  delayed
0            1   Rahul Kumar        1
2            3     Vijay Rao        1
3            4  Swathi Menon        1
1            2  Anita Sharma        0
4            5     Arjun Das        0
5            6    Pooja Nair        0


# STEP 9: Most Common Delivery Issues

In [ ]:
# Count frequency of each delivery status
issue_summary = merged_df['current_status'].value_counts()

print("Most Common Delivery Statuses:")
print(issue_summary)


Most Common Delivery Statuses:
current_status
Delivered     3
Shipped       2
In Transit    1
Pending       1
Name: count, dtype: int64


# Step 10: Display and save the cleaned data (Orders)

In [ ]:
# Display the cleaned orders data
print("\nCleaned Orders Data (after removing missing values):")
display(orders)

# Save the cleaned data to a new CSV
orders.to_csv("cleaned_orders.csv", index=False)




Cleaned Orders Data (after removing missing values):


,order_id,customer_id,order_date,delivery_date,status
0,1,1,2025-07-01,2025-07-03,Delivered
1,2,2,2025-07-05,2025-07-08,Delivered
2,3,3,2025-07-10,2025-07-12,Pending
3,4,4,2025-07-12,2025-07-15,Shipped
4,5,1,2025-07-18,2025-07-20,Pending
5,6,5,2025-07-20,2025-07-23,Delivered
6,7,6,2025-07-22,2025-07-25,Shipped


In [ ]:
# Download the file in Colab
from google.colab import files
files.download("cleaned_orders.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>